In [ ]:
import pandas as pd
import numpy as np
import re        
import glob
import os
from pathlib import Path
pd.options.display.max_rows = 999

# Directories

In [ ]:
userID = Path.home()
os.chdir(f'{userID}/Dropbox/Unison')

# Crosswalks

In [ ]:
file = glob.glob('data/xwalks/census_county_cbsa_2020.csv')[0]
cw_cbsa_county = pd.read_csv(file, encoding='latin-1', skiprows=2, skipfooter=4)
cw_cbsa_county = cw_cbsa_county[cw_cbsa_county['FIPS State Code']<=56]
cw_cbsa_county['cty_fips'] = cw_cbsa_county['FIPS State Code'] * 1000 + cw_cbsa_county['FIPS County Code']
cw_cbsa_county = cw_cbsa_county[['CBSA Code', 'CBSA Title', 'cty_fips']]
cw_cbsa_county.columns = ['cbsa_fips', 'cbsa_name', 'cty_fips']
print(f'Number of counties in crosswalk: {cw_cbsa_county.shape}')

# Add in Latitude/Longitude from County GAZ
file = glob.glob('data/xwalks/*Gaz_counties_national*.txt')[0]
cty_gaz = pd.read_csv('data/xwalks/2020_Gaz_counties_national.txt', encoding='latin-1', sep='\t')
cty_gaz.columns = [i.strip() for i in cty_gaz.columns]
cty_gaz = cty_gaz[['USPS', 'GEOID', 'INTPTLAT', 'INTPTLONG']]
cty_gaz.columns = ['state_name', 'cty_fips', 'lat', 'lon']
cty_gaz = cty_gaz[cty_gaz.state_name!="PR"]
cty_gaz['cty_fips'] = cty_gaz['cty_fips'].astype(int)
print(f'Number of counties in gazetteer: {cty_gaz.shape}')

# CBSA Name

In [ ]:
file = glob.glob('data/xwalks/census_cbsa_gaz2020_*.csv')[0]
cbsa_name = pd.read_csv(file, encoding='latin-1')
cbsa_name = cbsa_name[cbsa_name['CBSA_TYPE']==1]
cbsa_name['Area'] = cbsa_name['NAME'].apply(lambda x:x.replace(' Metro Area',''))
cbsa_name = cbsa_name[['GEOID', 'Area']].rename(columns={'GEOID':'Area FIPS Code'})
cbsa_name.shape

# State Name

In [ ]:
# State FIPS
file = glob.glob('data/xwalks/census_statefips*.csv')[0]
state_name = pd.read_csv(file, encoding='latin-1')
state_name.columns = ['Area', 'state_fips', 'state_abbrev']
state_name.head()

# Population

In [ ]:
# County 2020 Population
pop2000 = pd.read_csv('data/population/popest_0010_090121.csv', encoding='latin-1')
pop2000['state_fips'] = pop2000['STATE']
pop2000['cty_fips'] = 1000 * pop2000['state_fips'] + pop2000['COUNTY']
pop_cols = [i for i in pop2000.columns if re.findall('POPESTIMATE',i)]
pop_cols.remove('POPESTIMATE2010')
pop2000 = pop2000[['state_fips', 'SUMLEV', 'cty_fips'] + pop_cols]

# County 2020 Population
pop2020 = pd.read_csv('data/population/popest_1020_090121.csv', encoding='latin-1')
pop2020['state_fips'] = pop2020['STATE']
pop2020['cty_fips'] = 1000 * pop2020['state_fips'] + pop2020['COUNTY']
pop_cols = [i for i in pop2020.columns if re.findall('POPESTIMATE',i)]
for i in ['POPESTIMATE042020', 'POPESTIMATE2020']:
    pop_cols.remove(i)
pop2020 = pop2020[['state_fips','SUMLEV', 'cty_fips'] + pop_cols]

# County 2024 Population
files = glob.glob('data/population/popest_2024*.csv')[0]
pop2022 = pd.read_csv(files, encoding='latin-1')
pop2022['state_fips'] = pop2022['STATE']
pop2022['cty_fips'] = 1000 * pop2022['state_fips'] + pop2022['COUNTY']
pop_cols = [i for i in pop2022.columns if re.findall('POPESTIMATE',i)]
pop2022 = pop2022[['state_fips','SUMLEV', 'cty_fips'] + pop_cols]

# Merge 2000 to 2020 data
cty_pop = pd.merge(pop2000, pop2020, on=['state_fips', 'SUMLEV','cty_fips'])
cty_pop = cty_pop.merge(pop2022, on=['state_fips', 'SUMLEV','cty_fips'])

# Clean data
def clean_census(df, area_type):
    
    if area_type == 'MSA':
        pop = df[df.SUMLEV==50].copy()
        pop = pop.merge(cw_cbsa_county, on='cty_fips')
        pop = pop.rename(columns={'cbsa_name': 'Area'})
        pop.drop(columns=['cty_fips',  'cbsa_fips', 'state_fips', 'SUMLEV'], inplace=True)
        pop = pop.groupby(['Area']).sum()
    elif area_type == 'State':
        pop = df[df.SUMLEV==40].copy()
        pop = pop.merge(state_name, on='state_fips')
        pop.drop(columns=['cty_fips',  'state_fips', 'SUMLEV', 'state_abbrev'], inplace=True)
        pop = pop.groupby(['Area']).sum()
    elif area_type == 'Nation':
        pop = df[df.SUMLEV==40].copy()
        pop.drop(columns=['SUMLEV','state_fips','cty_fips'], inplace=True)
        pop = pop.sum().to_frame().reset_index()
        pop.columns = ['Year', 'Population']
        pop['Year'] = [i.replace('POPESTIMATE','') for i in pop['Year']]
        pop = pop.astype(int)
        pop['Area'] = 'United States'
    
    if area_type != "Nation":
        pop.columns = [i.replace('POPESTIMATE','') for i in pop.columns]
        pop = pop.reset_index().melt(id_vars='Area', var_name='Year', value_name='Population')
        pop = pop[['Area', 'Year', 'Population']]

    pop['Type'] = area_type
    pop.sort_values(by=['Area','Year'], inplace=True)
    pop.reset_index(inplace=True, drop=True)
    return pop

# MSA
msa = clean_census(cty_pop, 'MSA')
state = clean_census(cty_pop, 'State')
us = clean_census(cty_pop, 'Nation')

# Concatenate
pop = pd.concat([us, state, msa], axis=0, ignore_index=True)
pop['Date'] = pop['Year'].astype(str) + '-12-31'
pop['Date'] = pd.to_datetime(pop['Date'])
pop.drop(columns=['Year'], inplace=True)
pop.to_parquet(f'{userID}/Dropbox/Unison/Analytics/EconData/pop.parquet', index=False)
pop.head()

In [ ]:
pop.tail(12)

# BLS LN and LAU SERIES

In [ ]:
us_file = glob.glob(f'{userID}/Dropbox/Unison/data/lau/us_ur*')[0]
us = pd.read_csv(us_file, index_col=0, sep='\t')
us.columns

In [ ]:
# MSA Seasonally-Adjusted Unemployment Rate
file = glob.glob(f'{userID}/Dropbox/Unison/data/lau/msa_sur*.csv')[0]
data = pd.read_csv(file, skiprows=2, skipfooter=5)
data = data[~data['LAUS Code'].isnull()]
data = data[data['ST FIPS Code']<=56]
data = data[data['Month']<=12]
data = data[data['Year']>=2000]
for col in ['Year', 'Month']:
    data[col] = data[col].astype(int)
data['Month'] = data['Month'].astype(str)
data['Month'] = [f'0{i}' if len(i)==1 else i for i in data['Month']]
for col in ['Civilian Labor Force', 'Employment', 'Unemployment', 'Unemployment Rate']:
    data[col] = [str(i).replace(',','') for i in data[col]]
    data[col] = pd.to_numeric(data[col], errors='coerce')
data = data.rename(columns={'Civilian Labor Force':'Labor Force'})
data['Date'] = data['Year'].astype(str).values + data['Month'].values  + '01'
data['Date'] = pd.to_datetime(data['Date'], format='%Y%m%d') + pd.offsets.MonthEnd(0)
data = data[['Area', 'Date','Labor Force', 'Employment', 'Unemployment', 'Unemployment Rate']]
#data = data.groupby(['Area','Year']).mean().reset_index()
data['Type'] = ['MSA' if i.find('MSA')> -1 else 'NECTA' for i in data['Area']]
data['Area'] = [i.replace(' MSA','') for i in data['Area']]
data['Area'] = [i.replace(' Met NECTA','') for i in data['Area']]

#msa = data.merge(cbsa_name, on='Area FIPS Code', how='inner')
#msa.drop(columns=['BLS_Area','Area FIPS Code'], inplace=True)
msa = data.copy()
msa.tail()

most_recent_date = data['Date'].max()
print(f'({str(file)})')
print(f'Most recent date: {most_recent_date}')

In [ ]:
# US Unemployment
def clean_us_lau(lau, start=2000):

    '''
    LNS11000000 - Civilian Labor Force
    LNS12000000 - Employment
    LNS13000000 - Unemployment 
    '''

    us_dict = {
        'LNS11000000':'Labor Force',
        'LNS12000000':'Employment',
        'LNS13000000':'Unemployment',
        'LNS14000000':'Unemployment Rate'
    }

    df = lau.copy()
    df.reset_index(inplace=True)
    df.columns = [i.strip() for i in df.columns]
    df.series_id = df.series_id.str.strip()
    df = df[(df.series_id=='LNS11000000')|(df.series_id=='LNS12000000')|(df.series_id=='LNS13000000')|(df.series_id=='LNS14000000')]
    df = df.loc[df['period']!='M13']
    df = df[df.year>=start]
    df['month'] = [i.replace('M','') for i in df['period']]
    df['value'] = [str(i).replace('-','') for i in df['value']]
    df['value'] = df['value'].astype(float)
    df['Date'] = df['year'].astype(str).values + df['month'].values  + '01'
    df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d') + pd.offsets.MonthEnd(0)
    df = df[['series_id', 'Date', 'value']]
    df = df.groupby(['series_id', 'Date']).mean().reset_index()
    df['series_id'] = df['series_id'].replace(us_dict)
    df = df.pivot_table(index=['Date'], columns='series_id', values='value')
    df.reset_index(inplace=True)
    df.columns = ['Date', 'Employment', 'Labor Force', 'Unemployment', 'Unemployment Rate']
    df['Area'] = 'United States'
    df = df[['Area', 'Date', 'Employment', 'Labor Force', 'Unemployment', 'Unemployment Rate']]
    df[['Employment', 'Labor Force', 'Unemployment']] = (
        df[['Employment', 'Labor Force', 'Unemployment']].multiply(1000)
    )

    df['Type'] = 'Nation'
    #df['date'] = df['year'].astype(str) + '-01-01'
    #df['date'] = pd.to_datetime(df['date'], format="%Y-%m-%d")
    #df['variable'] = [str(i[-1]) for i in df.series_id]
    #df['value'] = [float(i) for i in df.value]
    return df

us_file = glob.glob(f'{userID}/Dropbox/Unison/data/lau/us_ur*')[0]
us = pd.read_csv(us_file, index_col=0, sep='\t')
us = clean_us_lau(us)
us.tail()

In [ ]:
# State Unemployment Statistics
lau_dict = {
    '3':'Unemployment Rate',
    '4':'Unemployment',
    '5':'Employment',
    '6':'Labor Force',
    '9':'Civilian Population'
}

def clean_state_lau(lau, start=1990):

    '''
    measure_code	measure_text
    03	unemployment rate
    04	unemployment
    05	employment
    06	labor force
    07	employment-population ratio
    08	labor force participation rate
    09	civilian noninstitutional population
    '''
    df = lau.copy()
    df.reset_index(inplace=True)
    df.columns = [i.strip() for i in df.columns]
    df.series_id = df.series_id.str.strip()
    df = df[df.year>=start]
    df['variable'] = [str(i[-1]) for i in df.series_id]
    df = df[(df['variable']=='3')|(df['variable']=='4')|(df['variable']=='5')|(df['variable']=='6')|(df['variable']=='9')]
    print(df.variable.unique())
    df['value'] = df['value'].apply(lambda x: str(x).strip().replace('-', ''))
    df['value'] = [float(i) if i!="" and i!="nan" else np.nan for i in df['value']]
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    
    # Debug: Check data types and values
    print(f"Value column dtype: {df['value'].dtype}")
    print(f"Non-numeric values: {df['value'].isna().sum()}")
    print(f"Sample values: {df['value'].head()}")
    
    df['variable'] = df['variable'].replace(lau_dict)
    df['seasonal'] = [i[4] for i in df.series_id]
    df['state_fips'] = [int(i[5:7]) for i in df.series_id]
    df['cty_fips'] = [int(i[5:10]) for i in df.series_id]
    df['preliminary'] = [1 if i=='P' else 0 for i in df['footnote_codes']]
    df = df[df['period']!='M13']
    df['month']=[i.replace('M','') for i in df['period']]
    df.drop(columns=['series_id', 'period', 'footnote_codes', 'preliminary', 'seasonal'], inplace=True)
    
    # Debug: Check data before pivot
    print(f"Data shape before pivot: {df.shape}")
    print(f"Value column dtype before pivot: {df['value'].dtype}")
    print(f"Unique variables: {df['variable'].unique()}")
    print(f"Sample data before pivot:")
    print(df.head())
    
    df = df.pivot_table(index=['cty_fips', 'state_fips', 'year', 'month'],  columns='variable', values='value', aggfunc='mean')
    df.reset_index(inplace=True)
    
    # Check the actual number of columns and assign names accordingly
    print(f"Actual number of columns: {len(df.columns)}")
    print(f"Column names: {list(df.columns)}")
    
    # Define expected column names
    expected_columns = ['cty_fips', 'state_fips', 'Year', 'Month', 'Employment', 'Labor Force', 'Unemployment','Unemployment Rate']
    
    # Only assign the number of columns that actually exist
    if len(df.columns) == len(expected_columns):
        df.columns = expected_columns
    else:
        # Create a mapping for the actual columns
        column_mapping = {}
        for i, col in enumerate(df.columns):
            if i < len(expected_columns):
                column_mapping[col] = expected_columns[i]
        df = df.rename(columns=column_mapping)
        print(f"Renamed columns to: {list(df.columns)}")
    df = df.merge(state_name, on='state_fips')
    df['Date'] = df['Year'].astype(str).values + df['Month'].values  + '01'
    df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d') + pd.offsets.MonthEnd(0)
    df = df[['Area', 'Date', 'Employment', 'Labor Force', 'Unemployment', 'Unemployment Rate']]
    #df = df.groupby(['Area', 'Date']).mean().reset_index().drop(columns=['state_fips', 'cty_fips'])
    df['Type'] = 'State'
    return df

state_file = glob.glob(f'{userID}/Dropbox/Unison/data/lau/state_ur*')[0]
state = pd.read_csv(state_file, index_col=0, sep='\t')
state = clean_state_lau(state)
state

In [ ]:
# Not merged NECTAs
# missing_bls = data.merge(cbsa_name, on='Area FIPS Code', how='left')
# missing_bls = missing_bls[missing_bls['Area'].isnull()]
# missing_bls['BLS_Area'].unique()

In [ ]:
msa.info()

In [ ]:
#Only save most recent quarter (month count =3)
lau = pd.concat([us, state, msa], axis=0, ignore_index=True)

lau['Quarter'] = lau['Date'].dt.to_period('Q')
df = lau[lau.Date<=most_recent_date].copy()
most_recent_Q = (
    df[
        (df['Employment'].groupby([df['Area'], df['Quarter']]).transform('count').eq(3))
    ]
    .Quarter
    .max()
)
most_recent_Q

In [ ]:
# Concatenate
use_most_recent_quarter = False
lau = pd.concat([us, state, msa], axis=0, ignore_index=True)
# lau = lau.loc[lau.Date<=most_recent_date]

#Only save most recent quarter (month count =3)
lau['Quarter'] = lau['Date'].dt.to_period('Q')
df = lau.loc[lau.Date<=most_recent_date].copy()
most_recent_Q = (
    df.loc[
        (df['Employment'].groupby([df['Area'], df['Quarter']]).transform('count').eq(3))
    ]
    .Quarter
    .max()
)
if use_most_recent_quarter:
    lau = lau[lau.Quarter<=most_recent_Q]

#Add all months, assuming December 2023 coming out soon
#lau['Quarter'] = lau['Quarter'].dt.to_timestamp()  + pd.offsets.QuarterEnd(0)

# Calculate 
lau.drop(columns='Quarter', inplace=True)
lau['Most Recent M'] = lau['Date'].max()
lau['Most Recent MSA M'] = most_recent_date # Based on MSA
lau['Most Recent Q'] = most_recent_Q
lau_monthly = lau.copy()
lau_monthly = lau_monthly.groupby(
    ['Area', 'Type', 'Most Recent Q', 'Most Recent M', 'Most Recent MSA M', pd.Grouper(key='Date', freq='Q')]
).mean().reset_index()
lau_monthly.to_parquet(f'{userID}/Dropbox/Unison/Analytics/EconData/lau.parquet', index=False)


lau = lau.groupby(
    ['Area', 'Type', 'Most Recent Q', 'Most Recent M', 'Most Recent MSA M', pd.Grouper(key='Date', freq='Y')]
).mean().reset_index()
lau.to_parquet(f'{userID}/Dropbox/Unison/Analytics/EconData/lau.parquet', index=False)
lau.tail()

In [ ]:
# # Not Seasonally-Adjusted (New Orleans data are affected because of Hurricane Katrina - need to be fixed.)
# lau_dict = {
#     '4':'Unemployment',
#     '5':'Employment',
#     '6':'Labor Force',
#     '9':'Civilian Population'
# }

# file = glob.glob(f'{userID}/Dropbox/Unison/Data/lau/county_ur_*.csv')[0]
# lau = pd.read_csv(file)
# lau['variable'] = [str(i[-1]) for i in lau.series_id]
# lau = lau[(lau['variable']=='4')|(lau['variable']=='5')|(lau['variable']=='6')|(lau['variable']=='9')]
# lau['value'] = lau['value'].apply(lambda x: str(x).replace('-',''))
# lau['value'] = [float(i) if i!="" else np.nan for i in lau['value']]
# lau['value'] = lau['value'].astype(float)
# lau['variable'] = lau['variable'].replace(lau_dict)
# lau['seasonal'] = [i[4] for i in lau.series_id]
# lau['state_fips'] = [int(i[5:7]) for i in lau.series_id]
# lau['cty_fips'] = [int(i[5:10]) for i in lau.series_id]
# lau['preliminary'] = [1 if i=='P' else 0 for i in lau['footnote_codes']]
# lau = lau[lau['period']!='M13']
# lau['month']=[i.replace('M','') for i in lau['period']]
# lau.drop(columns=['series_id', 'period', 'footnote_codes', 'preliminary'], inplace=True)
# lau = lau.pivot_table(index=['cty_fips', 'state_fips', 'year', 'month'],  columns='variable')
# lau.reset_index(inplace=True)
# lau.columns = ['cty_fips', 'state_fips', 'Year', 'Month', 'Employment', 'Labor Force', 'Unemployment']
# lau.head()
# def clean_lau(df, area_type):
#     ur = df.copy()
#     if area_type == 'MSA':
#         ur = ur.merge(cw_cbsa_county, on='cty_fips')
#         ur = ur.rename(columns={'cbsa_name': 'Area'})
#         ur.drop(columns=['cty_fips',  'cbsa_fips', 'state_fips'], inplace=True)
#         ur = ur.groupby(['Area','Year', 'Month']).sum().reset_index()
#         ur = ur.groupby(['Area','Year']).mean()
#     elif area_type == 'State':
#         ur = ur.merge(state_name, on='state_fips')
#         ur = ur.rename(columns={'state_name': 'Area'})
#         ur.drop(columns=['cty_fips',  'state_fips', 'state_abbrev'], inplace=True)
#         ur = ur.groupby(['Area','Year', 'Month']).sum().reset_index()
#         ur = ur.groupby(['Area','Year']).mean()
#     elif area_type == 'Nation':
#         ur.drop(columns=['cty_fips','state_fips'], inplace=True)
#         ur = ur.groupby(['Year', 'Month']).sum()
#         ur = ur.groupby(['Year']).mean()
#         ur['Area'] = 'United States'

#     ur['Type'] = area_type
#     ur.reset_index(inplace=True)
#     #ur.loc[ur.Preliminary>0, 'Preliminary'] = 1
#     ur.sort_values(by=['Area','Year'], inplace=True)
#     return ur

# # MSA
# msa = clean_lau(lau, 'MSA')
# state = clean_lau(lau, 'State')
# us = clean_lau(lau, 'Nation')

# # Concatenate
# ur = pd.concat([us, state, msa], axis=0, ignore_index=True)
# ur.to_csv(f'/Users/{userID}/Dropbox/Unison/Analytics/EconData/ur.csv', index=False)
# ur.head()

# BEA Data Series

### BEA Helper Functions

In [ ]:
cbsa_name = cw_cbsa_county[['cbsa_fips', 'cbsa_name']].copy()
cbsa_name.drop_duplicates(keep='first', inplace=True)

In [ ]:
# Helper functions
import sys
sys.path.append(f'{Path.home()}/Dropbox/Unison/Code')
from helper_functions import *

# API key path
api_key_path = f'{Path.home()}/Dropbox/Unison/Analytics'
api_key = get_key(f'{api_key_path}/api_keys.txt', 'fred')
extractor = FREDDataExtractor(api_key)

pcepi = extractor.get_series_data('PCEPI', 'PCEPI')
pcepi = pcepi.resample('Y').mean()
pcepi['PCEPI2017'] = pcepi.loc[pcepi.index.year==2017, 'PCEPI'].mean()
pcepi['Year'] = pcepi.index.year

def deflate(df, var):
    deflate_var=df[var].values / df['PCEPI'].values * df['PCEPI2017'].values
    return deflate_var


In [ ]:
# # BEA PCEPI
# pcepi_file = glob.glob(f'{userID}/Dropbox/Unison/Data/xwalks/pcepi_*.csv')
# pcepi = pd.read_csv(pcepi_file[0])
# pcepi['DATE'] = pd.to_datetime(pcepi['DATE'])
# pcepi['Year'] = pcepi['DATE'].dt.year
# pcepi = pcepi[pcepi.PCEPI!='.']
# pcepi['PCEPI']=pcepi['PCEPI'].astype(float)
# pcepi['Year']=pcepi['Year'].astype(int)
# pcepi = pcepi[['PCEPI', 'Year']]
# pcepi['PCEPI2017'] = pcepi.loc[pcepi.Year==2017, 'PCEPI'].values[0]
# pcepi.info()

# def deflate(df, var):
#     deflate_var=df[var].values / df['PCEPI'].values * df['PCEPI2017'].values
#     return deflate_var


In [ ]:
gdp_var = ['Real GDP', 'Real GDP Quality Index', 'Current-Dollar GDP']
inc_var = ['Personal Income', 'Population']

gdp_dict = {
    'Real GDP':1,
    'Real GDP Quality Index':2,
    'Current-Dollar GDP':3,
}
    
inc_dict = {
    'Personal Income':1,
    'Population':2
}

# Get Year Range
def get_year_range(file):
    file_years = re.findall('\d{4}', file)
    assert len(file_years)==2
    if int(min(file_years))<2001:
        file_start = 2001
    else:
        file_start = int(min(file_years))
    file_end = int(max(file_years))
    years = [str(i) for i in range(file_start, file_end + 1)]
    return years

# Define clean data
def clean_state_data(file, var_list, var_dict):
    df = pd.read_csv(file, skipfooter=4)
    df['GeoFIPS'] = [i.replace('"','') for i in df.GeoFIPS]
    df['GeoFIPS'] = df['GeoFIPS'].astype(int)
    df = df[df.GeoFIPS<=56000]
    year_list = get_year_range(file)
    cols = ['GeoName', 'LineCode', 'Unit'] + year_list
    df = df[cols]
    data = pd.DataFrame()
    for var_name in var_list:
        temp = df[df.LineCode==var_dict[var_name]].copy()
        temp.drop(columns='LineCode', inplace=True)
        temp = temp.melt(id_vars=['GeoName', 'Unit'], var_name='Year', value_name=var_name)
        temp[var_name] = [np.nan if i=='(NA)' else i for i in temp[var_name]]
        var_unit = temp['Unit'].values[0]
        if var_name == "Personal Income":
            if re.findall('Millions of', var_unit):
                temp[var_name] = temp[var_name].astype(float)
                temp[var_name] = temp[var_name].values*10**6 / 10**3
            if re.findall('Thousands of', var_unit):
                temp[var_name] = temp[var_name].astype(float)
            new_name = f'{var_name} (Thousands)'
            temp = temp.rename(columns={var_name: new_name})
        elif (var_name == "Real GDP") | (var_name == "Current-Dollar GDP"):
            if re.findall('Millions of', var_unit):
                temp[var_name] = temp[var_name].astype(float)
            if re.findall('Thousands of', var_unit):
                temp[var_name] = temp[var_name].astype(float)
                temp[var_name] = temp[var_name].values*10**3 / 10**6
            new_name = f'{var_name} (Millions)'
            temp = temp.rename(columns={var_name: new_name})
        else:
            if re.findall('Millions of', var_unit):
                temp[var_name] = temp[var_name].astype(float)
                new_name = f'{var_name} (Millions)'
                temp = temp.rename(columns={var_name: new_name})
                #temp[var_name] = temp[var_name].values*10**6
            if re.findall('Thousands of', var_unit):
                temp[var_name] = temp[var_name].astype(float)
                new_name = f'{var_name} (Thousands)'
                temp = temp.rename(columns={var_name: new_name})
                #temp[var_name] = temp[var_name].values*10**3
        temp.drop(columns='Unit', inplace=True)
        if data.shape[0]==0:
            data = pd.concat([data, temp], axis=1)
        else:
            data = pd.merge(data, temp, on=['GeoName', 'Year'])
    data = data.rename(columns={'GeoName':'Area'})
    data['Area'] = [i.replace(' *','') for i in data['Area']]
    return data

def get_states(gdp, inc):
    clean_gdp = clean_state_data(gdp[0], gdp_var, gdp_dict)
    clean_inc = clean_state_data(inc[0], inc_var, inc_dict)
    data = pd.merge(clean_gdp, clean_inc, on=['Area', 'Year'], how='outer')
    us = data[data.Area=='United States'].copy()
    us['Year']=us['Year'].astype(int)
    for var in ['Real GDP (Millions)', 'Real GDP Quality Index', 'Current-Dollar GDP (Millions)', 'Personal Income (Thousands)']:
        us[var]=us[var].astype(float)
    us = us.merge(pcepi, on='Year')
    us['Real Personal Income (Thousands)']=deflate(us, 'Personal Income (Thousands)')
    us.sort_values(by=['Area', 'Year'], inplace=True)
    us['Type'] = 'Nation'
    
    state = data[data.Area!='United States'].copy()
    state.sort_values(by=['Area', 'Year'], inplace=True)
    state['Type'] = 'State'
    state['Year']=state['Year'].astype(int)
    for var in ['Real GDP (Millions)', 'Real GDP Quality Index', 'Current-Dollar GDP (Millions)', 'Personal Income (Thousands)']:
        state[var]=state[var].astype(float)
    state = state.merge(pcepi, on='Year')
    state['Real Personal Income (Thousands)'] = deflate(state, 'Personal Income (Thousands)')

    final_data = pd.concat([us, state], axis=0, ignore_index=True)
    return final_data

# Clean GDP
gdp = glob.glob('data/bea/SAGDP1__ALL_AREAS_*.csv')
inc = glob.glob('data/bea/SAINC1__ALL_AREAS_*.csv')
state = get_states(gdp, inc)

In [ ]:
# Get MSA (Including US)
def clean_msa_data(file, var_list, var_dict):
    df = pd.read_csv(file, skipfooter=4, encoding='latin-1')
    for col in df.columns:
        df[col] = df[col].replace('(NA)', np.nan)
        df[col] = df[col].replace('(NM)', np.nan)
    df['GeoFIPS'] = [i.replace('"','') for i in df.GeoFIPS]
    df['GeoFIPS'] = df['GeoFIPS'].astype(int)
    df = df.rename(columns={'GeoName':'Area'})
    df['Area'] = [i.replace('(Metropolitan Statistical Area)','MSA') for i in df['Area']]
    df['Area'] = [i.split(' MSA')[0] for i in df['Area']]
    year_list = get_year_range(file)
    cols = ['Area', 'LineCode', 'Unit'] + year_list
    df = df[cols]
    data = pd.DataFrame()
    for var_name in var_list:
        temp = df[df.LineCode==var_dict[var_name]].copy()
        temp.drop(columns='LineCode', inplace=True)
        temp = temp.melt(id_vars=['Area', 'Unit'], var_name='Year', value_name=var_name)
        var_unit = temp['Unit'].values[0]
        if var_name == "Personal Income":
            if re.findall('Millions of', var_unit):
                temp[var_name] = temp[var_name].astype(float)
                temp[var_name] = temp[var_name].values*10**6 / 10**3
            if re.findall('Thousands of', var_unit):
                temp[var_name] = temp[var_name].astype(float)
            new_name = f'{var_name} (Thousands)'
            temp = temp.rename(columns={var_name: new_name})
        elif (var_name == "Real GDP") | (var_name == "Current-Dollar GDP"):
            if re.findall('Millions of', var_unit):
                temp[var_name] = temp[var_name].astype(float)
            if re.findall('Thousands of', var_unit):
                temp[var_name] = temp[var_name].astype(float)
                temp[var_name] = temp[var_name].values*10**3 / 10**6
            new_name = f'{var_name} (Millions)'
            temp = temp.rename(columns={var_name: new_name})
        else:
            if re.findall('Millions of', var_unit):
                temp[var_name] = temp[var_name].astype(float)
                new_name = f'{var_name} (Millions)'
                temp = temp.rename(columns={var_name: new_name})
                #temp[var_name] = temp[var_name].values*10**6
            if re.findall('Thousands of', var_unit):
                temp[var_name] = temp[var_name].astype(float)
                new_name = f'{var_name} (Thousands)'
                temp = temp.rename(columns={var_name: new_name})
                #temp[var_name] = temp[var_name].values*10**3
        temp.drop(columns='Unit', inplace=True)
        if data.shape[0]==0:
            data = pd.concat([data, temp], axis=1)
        else:
            data = pd.merge(data, temp, on=['Area', 'Year'])
    return data


def get_msa(gdp, inc):
    # Clean GDP
    clean_gdp = clean_msa_data(gdp[0], gdp_var, gdp_dict)
    clean_inc = clean_msa_data(inc[0], inc_var, inc_dict)
    data = pd.merge(clean_gdp, clean_inc, on=['Area', 'Year'], how='outer')
    data['Type']='MSA'
    data['Year']=data['Year'].astype(int)
    for var in ['Real GDP (Millions)', 'Real GDP Quality Index', 'Current-Dollar GDP (Millions)', 'Personal Income (Thousands)']:
        data[var]=data[var].astype(float)
    data = data.merge(pcepi, on='Year')
    data['Real Personal Income (Thousands)'] = deflate(data, 'Personal Income (Thousands)')
    data.sort_values(by=['Area', 'Year'], inplace=True)
    return data


gdp = glob.glob('data/bea/CAGDP1_MSA_*.csv')
inc = glob.glob('data/bea/CAINC1_MSA_*.csv')
msa = get_msa(gdp, inc)

bea = pd.concat([state, msa], axis=0, ignore_index=True)
bea['Date'] = bea['Year'].astype(str) + '-12-31'
bea.drop(columns=['Year'], inplace=True)
bea = bea.sort_values(by=['Area', 'Date'])
bea.drop(columns=['PCEPI',	'PCEPI2017', 'Real GDP Quality Index',	'Current-Dollar GDP (Millions)', 'Personal Income (Thousands)'], inplace=True)
bea['Real Personal Income per Capita'] = bea['Real Personal Income (Thousands)'].values / bea['Population'].values * 1e3
bea['Real Personal Income per Capita'] = bea['Real Personal Income per Capita'].round(1)
bea['Real GDP per Capita'] = bea['Real GDP (Millions)'].values / bea['Population'].values * 1e6
bea = bea[[
    'Area', 'Type', 'Date', 
    'Real GDP (Millions)', 'Real GDP per Capita', 
    'Population', 'Real Personal Income (Thousands)',
    'Real Personal Income per Capita'
    ]]
bea.columns = [
    'Area', 'Type', 'Date', 'Real GDP', 
    'Real GDP per Capita', 'Population', 'Real Personal Income', 
    'Real Personal Income per Capita'
]
#bea.to_csv(f'/Users/{userID}/Dropbox/Unison/Analytics/EconData/bea.csv', index=False)

#bea_hist = bea.copy()
#bea_hist.head()
bea.to_parquet(f'{userID}/Dropbox/Unison/Analytics/EconData/bea.parquet', index=False)
bea.head()


In [ ]:
# # New Bea Data
# def get_states_2022(gdp):
#     data = clean_state_data(gdp[0], gdp_var, gdp_dict)
#     us = data[data.Area=='United States'].copy()
#     us['Year']=us['Year'].astype(int)
#     for var in ['Real GDP (Millions)', 'Real GDP Quality Index', 'Current-Dollar GDP (Millions)']:
#         us[var]=us[var].astype(float)
#     us = us.merge(pcepi, on='Year')
#     us.sort_values(by=['Area', 'Year'], inplace=True)
#     us['Type'] = 'Nation'
    
#     state = data[data.Area!='United States'].copy()
#     state.sort_values(by=['Area', 'Year'], inplace=True)
#     state['Type'] = 'State'
#     state['Year']=state['Year'].astype(int)
#     for var in ['Real GDP (Millions)', 'Real GDP Quality Index', 'Current-Dollar GDP (Millions)']:
#         state[var]=state[var].astype(float)
#     state = state.merge(pcepi, on='Year')

#     final_data = pd.concat([us, state], axis=0, ignore_index=True)
#     return final_data

# def get_msa_2022(gdp):
#     data = clean_msa_data(gdp[0], gdp_var, gdp_dict)
#     data['Type']='MSA'
#     data['Year']=data['Year'].astype(int)
#     for var in ['Real GDP (Millions)', 'Real GDP Quality Index', 'Current-Dollar GDP (Millions)']:
#         data[var]=data[var].astype(float)
#     data = data.merge(pcepi, on='Year')
#     data.sort_values(by=['Area', 'Year'], inplace=True)
#     return data

# state = glob.glob('data/bea/SAGDP1__ALL_AREAS_2017_2022.csv')
# msa = glob.glob('data/bea/CAGDP1_MSA_2017_2022.csv')

# state_2022 = get_states_2022(state)
# msa_2022 = get_msa_2022(msa)

# bea = pd.concat([state_2022, msa_2022], axis=0, ignore_index=True)
# bea['Date'] = bea['Year'].astype(str) + '-12-31'
# bea.drop(columns=['Year'], inplace=True)
# bea = bea.sort_values(by=['Area', 'Date'])
# bea.drop(columns=['PCEPI',	'PCEPI2017', 'Real GDP Quality Index',	'Current-Dollar GDP (Millions)'], inplace=True)
# bea = bea[['Area', 'Type', 'Date', 'Real GDP (Millions)']]
# bea.columns = ['Area', 'Type', 'Date', 'Real GDP (New)']
# bea_merged = bea_hist.merge(bea, on=['Area', 'Type', 'Date'], how='outer')
# bea_merged = bea_merged[['Area', 'Type', 'Date', 'Real GDP (Legacy)', 'Real GDP (New)', 'Real Personal Income per Capita']]
# bea_merged.to_parquet(f'{userID}/Dropbox/Unison/Analytics/EconData/bea.parquet', index=False)
# bea_merged.head()

# QCEW

In [ ]:
# Xwalks
file = glob.glob('data/xwalks/qcew_county_cbsa_*.csv')[0]
cw_qcew = pd.read_csv(file, encoding='latin-1', skipfooter=3)
cw_qcew.dropna(subset=['MSA Title'], inplace=True)
cw_qcew.loc[cw_qcew['MSA Title'].str.contains('MSA'), 'MSA Type'] = 'Metro'
cw_qcew

In [ ]:
# Xwalks
file = glob.glob('data/xwalks/qcew_county_cbsa_*.csv')[0]
cw_qcew = pd.read_csv(file, encoding='latin-1', skipfooter=3)
cw_qcew.dropna(subset=['MSA Title'], inplace=True)
cw_qcew.loc[cw_qcew['MSA Title'].str.contains('MSA'), 'MSA Type'] = 'Metro'
cw_qcew = cw_qcew[cw_qcew['MSA Type']=='Metro']
cw_qcew = cw_qcew[~cw_qcew['MSA Title'].str.contains(', PR')]
cw_qcew = cw_qcew[['County Code', 'MSA Code', 'MSA Title']]
cw_qcew['County Code'] = cw_qcew['County Code'].astype(int)
cw_qcew = cw_qcew[cw_qcew['County Code']<=56999]
cw_qcew['County Code'] = cw_qcew['County Code'].astype(str).str.zfill(5)
cw_qcew

In [ ]:
# Convert CSV files to Excel
convert = False
def excel_to_csv(file):
    df = pd.read_excel(file)
    file_name = re.findall('data/(.*)\.xlsx', file)[0]
    df.to_csv(f'data/{file_name}.csv', index=False)
    return df

if convert:
    for year in range(2024,2025):
        qcew_list = glob.glob(f'data/qcew/{year}_all_county_high_level/allhlcn*.xlsx')
        for file in qcew_list:
            excel_to_csv(file)


In [ ]:
files = glob.glob('data/qcew/*')
files = [''.join(re.findall('\d',i)) for i in files]
files = [int(i) for i in files]
min_year = min(files)
max_year = max(files)

quarter_dict = {'Q1':'03', 'Q2':'06', 'Q3':'09', 'Q4':'12'}

qcew = pd.DataFrame()
most_recent_q = None
for year in range(min_year, max_year+1):
    qcew_list = glob.glob(f'data/qcew/{year}_all_county_high_level/allhlcn???.csv')
    year_str = str(year)[-2:]
    for file in qcew_list:
        qt = file.replace('.csv','')[-1:]
        df = pd.read_csv(file)
        if "Area\r\nCode" in df.columns:
            df = df.rename(columns={"Area\r\nCode": "Area Code"})
        if "Area\nCode" in df.columns:
            df = df.rename(columns={"Area\nCode": "Area Code"})

        # Private, Total Industries
        df = df[(df['Own']==5) & (df['NAICS']==10)]
        df = df[['Area Code', 'Area', 'Area Type', 'St', 'Cnty', 'Establishment Count', 'St Name']]
        df.columns = ['Code', 'Area', 'Type', 'St', 'Cnty', 'Establishment', 'St Name']
        df['Area'] = df['Area'].astype(str)
        df['Code'] = df['Code'].astype(str).str.zfill(5)
        # df.loc[df.St=='US', 'St'] = '0'
        # df['St'] = df['St'].astype(int)
        # df['Cnty'] = df['Cnty'].astype(int) 
        # df['cty_fips'] = df['St'].astype(str) + '00' + df['Cnty'].astype(str)
        df['Establishment'] = [str(i).replace(',','') for i in df['Establishment']]
        df['Establishment'] = df['Establishment'].astype(float)
        df['Year'] = year
        df['Quarter'] = f'Q{qt}'
        
        us = df[df['Type']=='Nation'].copy()
        us['Area'] = 'United States'
        index_cols = ['Area', 'Type', 'Year', 'Quarter']
        cols = index_cols + ['Establishment']
        us = us[cols]
        us = us.groupby(index_cols).mean().reset_index()
       
        state = df[df['Type']=='State'].copy()
        state['Area'] = state['St Name'].values
        state = state[cols]
        state = state.groupby(index_cols).mean().reset_index()
       
        # MSA
        #msa = df[df['Type']=='MSA'].copy()
        #msa['Area'] = [i.replace('MSA','').strip() for i in msa['Area']]
        #msa = msa[cols]
        #msa = msa.groupby(index_cols).mean().reset_index()

        # Use county since geography changed starting 2024
        cty = df[df['Type']=='County'].copy()
        cty['County Code'] = cty['Code'].str.zfill(5)
        cty = cty.merge(cw_qcew, on='County Code')
        cty.drop(columns=['Code', 'Area', 'MSA Code'], inplace=True)
        cty = cty.rename(columns={'MSA Title':'Area'})
        cty['Type'] = 'MSA'
        cty = cty[cols]
        cty = cty.groupby(index_cols).sum().reset_index()
        
        #df = pd.concat([us, state, msa], axis=0, ignore_index=True)
        df = pd.concat([us, state, cty], axis=0, ignore_index=True)
        qcew = pd.concat([qcew, df], axis=0, ignore_index=True)

qcew['Date'] = qcew['Year'].astype(str) + qcew['Quarter'].replace(quarter_dict).astype(str)
qcew['Date'] = pd.to_datetime(qcew['Date'], format='%Y%m')
qcew['Most Recent Quarter'] = qcew['Date'].max()
qcew['Most Recent Quarter'] = qcew['Most Recent Quarter'].dt.to_period('Q')
qcew['Date'] = qcew['Date'] + pd.offsets.MonthEnd(0)
qcew.drop(columns=['Year', 'Quarter'], inplace=True)
qcew['Establishment'] = qcew['Establishment'].round(2)
qcew.sort_values(by=['Area', 'Type', 'Date', 'Most Recent Quarter'], inplace=True)
qcew.to_parquet(f'{userID}/Dropbox/Unison/Analytics/EconData/qcew.parquet', index=False)

# Industry employment

In [ ]:
cbsa_name = cw_cbsa_county[['cbsa_name', 'cbsa_fips']].drop_duplicates(keep='first')

file = glob.glob('data/ces/sm_all*')[0]

def clean_SM(data):

    '''
    Positions - Value - Field Name
    1-2 SM - Prefix (SM refers to State and metropolitan area series)
    3 - U - Seasonal Adjustment Code (S is for seasonally adjusted, U for unadjusted)
    4-5 - 08 - State Code
    6-10 - 00000 - Area Code
    11-12 - 43 - Supersector Code
    13-18 - 220000 - Industry Code **
    19-20 - 01 - Data Type
    
    data_type_code - data_type_text
    01 - All Employees, In Thousands
    02 - Average Weekly Hours of All Employees
    03 - Average Hourly Earnings of All Employees, In Dollars
    06 - Production or Nonsupervisory Employees, In Thousands
    07 - Average Weekly Hours of Production Employees
    08 - Average Hourly Earnings of Production Employees, In Dollars
    11 - Average Weekly Earnings of All Employees, In Dollars
    21 - Diffusion Indexes, 1-month span, seasonally adjusted, total nonfarm
    22 - Diffusion Indexes, 3-month span, seasonally adjusted, total nonfarm
    23 - Diffusion Indexes, 6-month span, seasonally adjusted, total nonfarm
    24 - Diffusion Indexes, 12-month span, seasonally adjusted, total nonfarm
    26 - All Employees, 3-month average change, In Thousands, seasonally adjusted
    30 - Average Weekly Earnings of Production Employees, In Dollars
    '''    
    var_dict = {
        '01': 'Employment (Thousands)',
        '02': 'Average Weekly Hours',
        '03': 'Average Hourly Earnings'
    }
    
    supersector_dict = {
        '00000000':'Total Nonfarm',
        #'10000000':'Mining and Logging',
        #'20000000':'Construction',
        '15000000':'Mining, Logging, and Construction',
        '30000000':'Manufacturing',
        '40000000':'Trade, Transportation, and Utilities',
        '50000000':'Information',
        '55000000':'Financial Activities', 
        '60000000':'Professional and Business Services',
        '65000000':'Education and Health Services',
        '70000000':'Leisure and Hospitality',
        '80000000':'Other Services',
        '90000000':'Government'
        #'100':'Mining, Logging, and Construction'
    }
    
    data = pd.read_csv(file, sep='\t')
    data.columns = [i.strip() for i in data.columns]
    data = data.loc[data['period']!='M13']
    data['series_id'] = [str(i).strip() for i in data['series_id']]
    data['Variable'] = [i[-2:] for i in data['series_id']]
    data['Seasonal'] = [i[2] for i in data['series_id']]
    data = data[data['Seasonal']=='U']
    data['state_fips'] = [i[3:5] for i in data['series_id']]
    data = data[(data.Variable=='01')]
    data['Variable'] = data['Variable'].replace(var_dict)
    data['Month'] = [i.replace('M','') for i in data['period']]
    data['Date'] = data['year'].astype(str) + '-' + data['Month'] + '-01'
    data['Date'] = pd.to_datetime(data['Date']) + pd.offsets.MonthEnd(0)
    data = data[data.Date.dt.year>=2000]
    data.drop(columns='year', inplace=True)
    data['cbsa_fips'] = [i[5:10] for i in data['series_id']]
    data['Supersector'] = [i[10:18] for i in data['series_id']]
    #data.loc[data.Supersector=='10', 'Supersector'] = '100'
    #data.loc[data.Supersector=='20', 'Supersector'] = '100'
    data['Supersector'] = data['Supersector'].replace(supersector_dict)
    data = data[data['Supersector'].isin(list(supersector_dict.values()))]
    data['value'] = [np.nan if str(i).strip()=='-' else i for i in data['value']]
    data['value'] = data['value'].astype(float)
    data = data[['cbsa_fips', 'state_fips', 'Seasonal', 'Variable', 'Date', 'Supersector', 'value']]
    data = data.groupby(['cbsa_fips', 'state_fips', 'Seasonal', 'Variable', 'Date', 'Supersector']).mean()
    data.reset_index(inplace=True)
    data = data.rename(columns={'value':'Employment (Thousands)'})
    #data = data.pivot_table(index=['cbsa_fips', 'state_fips', 'Seasonal', 'Date' , 'Supersector'], columns=['Variable'])
    #data.columns = ['Employment (Thousands)']
    data.reset_index(inplace=True)

    data['cbsa_fips'] = data['cbsa_fips'].astype(int)
    data['state_fips'] = data['state_fips'].astype(int)
    data = data[data['state_fips']<=56]
    data['Type'] = 'MSA'
    data.loc[data['cbsa_fips']==0, 'Type'] = 'State'
    data = data.merge(cbsa_name, on='cbsa_fips', how='left')
    data['state_fips'] = data['state_fips'].astype(int)
    data = data.merge(state_name, on='state_fips', how='left')
    data.loc[data['Type']=='State', 'cbsa_name'] = data.loc[data['Type']=='State', 'Area']
    data.drop(columns=['cbsa_fips', 'state_abbrev', 'state_fips', 'Area', 'Seasonal', 'Variable'], inplace=True)
    data = data.rename(columns={'cbsa_name':'Area'})
    data['Employment'] = data['Employment (Thousands)'].multiply(1000).round(0)
    data = data[['Area', 'Type', 'Date', 'Supersector', 'Employment']]
    data = data[~data['Area'].isnull()]
    return data

sm = clean_SM(file)
sm.sort_values(by=['Area', 'Type', 'Supersector', 'Date'], inplace=True)
#sm.to_parquet(f'{userID}/Dropbox/Unison/Analytics/EconData/sm.parquet', index=False)
sm.head()

In [ ]:
sm_pivot = sm.copy()
sm_pivot = sm_pivot[sm_pivot.Date.dt.year==sm_pivot.Date.dt.year.max()]
sm_pivot = sm_pivot.pivot_table(index=['Area', 'Type', 'Date'], columns='Supersector', values='Employment').reset_index()
sm_pivot['Quarter'] = sm_pivot['Date'].dt.to_period('Q')
sm_pivot = sm_pivot[sm_pivot[['Area', 'Quarter','Total Nonfarm']].groupby(['Area', 'Quarter']).transform('count').eq(3)['Total Nonfarm']]
sm_pivot = sm_pivot[sm_pivot.Quarter == sm_pivot.Quarter.max()]
sm_pivot['Most Recent'] = sm_pivot['Date'].max()
sm_pivot = sm_pivot.melt(id_vars=['Area', 'Type', 'Date', 'Quarter', 'Most Recent'], var_name='Supersector', value_name='Employment')
sm_pivot = sm_pivot.groupby(['Area', 'Type', 'Supersector', pd.Grouper(key='Date', freq='Q')]).mean().reset_index()
sm_pivot['Format'] = 'Quarter'
sm_pivot['Most Recent'] = sm_pivot['Date']
sm_pivot
sm_agg = sm.copy()
sm_agg['Most Recent'] = sm_agg['Date'].max()
sm_agg = sm_agg.groupby(['Area', 'Type', 'Supersector', 'Most Recent', pd.Grouper(key='Date', freq='Y')]).mean().reset_index()
sm_agg['Format'] = 'Year'
ces_all = pd.concat([sm_pivot, sm_agg], axis=0, ignore_index=True)
ces_all.sort_values(
    by = ['Format', 'Area', 'Type', 'Supersector', 'Date'],
)

In [ ]:
ces_data = glob.glob('data/ces/ce_*')[0]

def cleanCES(data, start, end):
    data = pd.read_csv(data,  sep='\t')
    data.columns = [i.strip() for i in data.columns]
    data['series_id'] = data['series_id'].str.strip()
    data = data.loc[(data.year>=start) & (data.year<=end)]
    data = data.loc[data['period']!='M13']
    data['month'] = data['period'].str.replace('M', '')
    data['value'] = data['value'] * 1000
    data['Date'] = data['year'].astype(str) + '-' + data['month'] + '-01'
    data['Date'] = pd.to_datetime(data['Date'])
    data['Date'] = data['Date'] +  pd.offsets.MonthEnd(0)
    data.set_index('Date', inplace=True)
    return data[['series_id', 'value']]

def getCESseries(data, naics):
    series = 'CE'
    seasonal = 'U'
    naics = naics
    industry = '0' * 6
    data_type = '01'
    
    series_id = series + seasonal + naics + industry + data_type   
    
    data = data[data.series_id==series_id]
    new_data = data.drop(columns='series_id').copy()
    print(series_id)
    return new_data

def getCES(data, start=2000, end=2024):
    start = start
    end = end
    clean_data = cleanCES(data, start, end)
    supersector_dict = {
        'Total Nonfarm' : '00',
        'Mining and Logging' : '10',
        'Construction': '20',
        'Manufacturing' : '30',
        'Trade, Transportation, and Utilities' : '40',
        'Information' : '50',
        'Financial Activities' : '55', 
        'Professional and Business Services' : '60',
        'Education and Health Services' : '65',
        'Leisure and Hospitality' : '70',
        'Other Services' : '80',
        'Government' : '90'
    }
    
    CESseries = pd.DataFrame()
    for name, supersector in supersector_dict.items():
        naics = supersector
        series = getCESseries(clean_data, naics)
        series.columns = [name]
        CESseries = pd.concat([CESseries, series], axis=1)
        
    CESseries['Mining, Logging, and Construction'] = CESseries['Mining and Logging'] + CESseries['Construction']
    CESseries.drop(columns=['Mining and Logging', 'Construction'], inplace=True)
    CESseries.reset_index(inplace=True)
    CESseries['Area'] = 'United States'
    CESseries['Type'] = 'Nation'
    return CESseries

US_ces = getCES(ces_data)
US_ces.head()

In [ ]:
sm_pivot = US_ces.copy()
sm_pivot = sm_pivot[sm_pivot.Date.dt.year==sm_pivot.Date.dt.year.max()]
sm_pivot['Quarter'] = sm_pivot['Date'].dt.to_period('Q')
sm_pivot = sm_pivot[sm_pivot[['Area', 'Quarter','Total Nonfarm']].groupby(['Area', 'Quarter']).transform('count').eq(3)['Total Nonfarm']]
sm_pivot = sm_pivot[sm_pivot.Quarter == sm_pivot.Quarter.max()]
sm_pivot['Most Recent'] = sm_pivot['Date'].max()
sm_pivot = sm_pivot.melt(id_vars=['Area', 'Type', 'Date', 'Quarter', 'Most Recent'], var_name='Supersector', value_name='Employment')
sm_pivot = sm_pivot.groupby(['Area', 'Type', 'Supersector', pd.Grouper(key='Date', freq='Q')]).mean().reset_index()
sm_pivot['Format'] = 'Quarter'
sm_pivot['Most Recent'] = sm_pivot['Date']
sm_pivot
sm_agg = US_ces.copy()
sm_agg['Most Recent'] = sm_agg['Date'].max()
sm_agg = sm_agg.melt(id_vars=['Area', 'Type', 'Date', 'Most Recent'], var_name='Supersector', value_name='Employment')
sm_agg = sm_agg.groupby(['Area', 'Type', 'Supersector', 'Most Recent', pd.Grouper(key='Date', freq='Y')]).mean().reset_index()
sm_agg['Format'] = 'Year'
sm_all = pd.concat([sm_pivot, sm_agg], axis=0, ignore_index=True)
sm_all.sort_values(
    by = ['Format', 'Area', 'Type', 'Supersector', 'Date'],
)

ces_all = pd.concat([ces_all, sm_all], axis=0, ignore_index=True)
ces_all.sort_values(
    by = ['Format', 'Area', 'Type', 'Supersector', 'Date'],
    inplace=True
)
ces_all.to_parquet(f'{userID}/Dropbox/Unison/Analytics/EconData/ces_all.parquet', index=False)

In [ ]:
ces_all.tail()